In [58]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [59]:
df = pd.DataFrame({
    "user_id": [22, 35, 28, 40, 19, 33, 30, 27, 45, 23],
    "user_gender": ['M', 'F', 'M', 'F', 'M', 'F', 'M', 'M', 'M', 'F'],
    "user_country": ['US', 'IN', 'UK', 'US', 'IN', 'FR', 'DE', 'IN', 'US', 'UK'],
    "ad_id": ['A1', np.nan, 'A1', 'A3', 'A2', 'A1', 'A4', 'A3', 'A2', 'A1'],
    "ad_type": ['banner', 'video', 'banner', np.nan, 'video', 'banner', 'native', 'video', 'banner', 'native'],
    "bid_price": [0.25, 0.90, 0.40, 0.55, 1.10, 0.30, 0.75, 0.95, np.nan, 0.60],
    "device": ['mobile', 'desktop', 'mobile', 'tablet', 'mobile', 'desktop', 'mobile', 'tablet', 'mobile', 'desktop'],
    "hour": [10, 18, 14, 20, 9, 16, 12, 19, 11, 21],
    "day_of_week": [1, 4, 2, 0, 3, 1, 6, 2, 4, 1],
    "prev_click": [0, 1, 0, 2, 0, 1, 1, 0, 0, 2],
    "impression": [3, 5, 2, 6, 4, 5, 3, 4, 2, 7],
    "clicked": [0, 1, 0, 1, 0, 1, 0, 0, 0, 1]
})

df

,user_id,user_gender,user_country,ad_id,ad_type,bid_price,device,hour,day_of_week,prev_click,impression,clicked
0,22,M,US,A1,banner,0.25,mobile,10,1,0,3,0
1,35,F,IN,NaN,video,0.90,desktop,18,4,1,5,1
2,28,M,UK,A1,banner,0.40,mobile,14,2,0,2,0
3,40,F,US,A3,NaN,0.55,tablet,20,0,2,6,1
4,19,M,IN,A2,video,1.10,mobile,9,3,0,4,0
5,33,F,FR,A1,banner,0.30,desktop,16,1,1,5,1
6,30,M,DE,A4,native,0.75,mobile,12,6,1,3,0
7,27,M,IN,A3,video,0.95,tablet,19,2,0,4,0
8,45,M,US,A2,banner,NaN,mobile,11,4,0,2,0
9,23,F,UK,A1,native,0.60,desktop,21,1,2,7,1


In [60]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

df[num_cols] = df[num_cols].fillna(df[num_cols].mean())

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])


In [61]:
y = df['clicked']
X = df.drop(columns=['clicked'], axis=1)

In [62]:
X

,user_id,user_gender,user_country,ad_id,ad_type,bid_price,device,hour,day_of_week,prev_click,impression
0,22,M,US,A1,banner,0.250000,mobile,10,1,0,3
1,35,F,IN,A1,video,0.900000,desktop,18,4,1,5
2,28,M,UK,A1,banner,0.400000,mobile,14,2,0,2
3,40,F,US,A3,banner,0.550000,tablet,20,0,2,6
4,19,M,IN,A2,video,1.100000,mobile,9,3,0,4
5,33,F,FR,A1,banner,0.300000,desktop,16,1,1,5
6,30,M,DE,A4,native,0.750000,mobile,12,6,1,3
7,27,M,IN,A3,video,0.950000,tablet,19,2,0,4
8,45,M,US,A2,banner,0.644444,mobile,11,4,0,2
9,23,F,UK,A1,native,0.600000,desktop,21,1,2,7


In [63]:
y

,clicked
0,0
1,1
2,0
3,1
4,0
5,1
6,0
7,0
8,0
9,1


In [64]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [65]:
enc = OneHotEncoder(handle_unknown='ignore')
X_train_enc = enc.fit_transform(X_train)
X_test_enc = enc.transform(X_test)

In [66]:
#help(DecisionTreeClassifier)

In [67]:
decision_tree = DecisionTreeClassifier(criterion='gini',max_depth=10,min_samples_split=30,random_state=42)
decision_tree.fit(X_train_enc, y_train)

y_pred = decision_tree.predict(X_test_enc)
y_proba = decision_tree.predict_proba(X_test_enc)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))


Accuracy: 0.5
ROC AUC: 0.5


In [68]:
parameters = {
    'criterion' : ['gini', 'entropy', 'log_loss'],
    'splitter' : ['best', 'random'],
     'max_features' : ['sqrt', 'log2'],
    'max_depth': [3, 10, None],
    'min_samples_split': [30, 40, 50]
}

grid_search = GridSearchCV(decision_tree,parameters,cv=3,scoring='roc_auc',n_jobs=-1)

grid_search.fit(X_train_enc, y_train)
print("Best parameters:", grid_search.best_params_)
print("Best ROC AUC score:", grid_search.best_score_)

Best parameters: {'criterion': 'gini', 'max_depth': 3, 'max_features': 'sqrt', 'min_samples_split': 30, 'splitter': 'best'}
Best ROC AUC score: 0.5


In [69]:
feature_names = enc.get_feature_names_out()
coefs = grid_search.best_estimator_.feature_importances_

sorted_idx = np.argsort(np.abs(coefs))[::-1]

print("Most important features:", feature_names[sorted_idx[:10]])
print("\nLeast important features:", feature_names[sorted_idx[-10:]])

Most important features: ['impression_7' 'impression_6' 'impression_5' 'impression_4'
 'impression_3' 'impression_2' 'prev_click_2' 'prev_click_1'
 'prev_click_0' 'day_of_week_6']

Least important features: ['user_gender_M' 'user_gender_F' 'user_id_40' 'user_id_33' 'user_id_30'
 'user_id_28' 'user_id_27' 'user_id_23' 'user_id_22' 'user_id_19']
